# Modeling — LSTM (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-19-lstm-modeling-design.md`.
Desember 2025 tidak dibuka di notebook ini.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch

from utils.modelling import evaluation, model_lstm as lstm
from utils.modelling import modeling_prep, walk_forward

df = pd.read_parquet(modeling_prep.MODEL_INPUT_FILE)
print(df.shape, torch.__version__, torch.backends.mps.is_available())
print(f"QUANTILE_SET: {len(lstm.QUANTILES)} titik, "
      f"{lstm.QUANTILES[0]}..{lstm.QUANTILES[-1]}")


## Benchmark

Satu putaran dua-fit di fold 5 dengan `DEFAULT_PARAMS`, di CPU.
MPS tidak diukur ujung-ke-ujung: probe 15 batch di fold 5
(2026-08-19) mencatat 0.392 s/batch di MPS lawan 0.193 s/batch di
CPU — tidak ada kernel LSTM ter-fusi di MPS pada hidden size ini,
jadi menjalankan benchmark penuh di sana hanya menghabiskan ~4 jam
untuk mengonfirmasi perangkat yang sudah kalah 2x.

Yang diukur: detik per epoch, epoch tempat early stopping mendarat,
dan peak RSS. Ketiganya yang mengisi rumus anggaran di §2.2 spec.

In [ ]:
import resource, time

frame = walk_forward.eligible_rows(df)
split = walk_forward.prepare_fold(frame, 5, prepared=True)
print('train', len(split['train']), 'valid', len(split['valid']))

benchmark = {}
for device_name in ('cpu',):
    try:
        make = lstm.bind_panel(df, device_name=device_name)
    except ValueError as failure:
        print(device_name, 'dilewati:', failure)
        continue
    fit_predict = make(lstm.DEFAULT_PARAMS, quantiles=lstm.QUANTILES)
    started = time.time()
    prediction = fit_predict(split['train'], split['valid'])
    elapsed = time.time() - started
    best_epoch = fit_predict.best_epochs[0]
    # elapsed covers both fits: best_epoch + patience epochs in the
    # first, best_epoch in the second.
    epochs_run = 2 * best_epoch + lstm.EARLY_STOPPING_EPOCHS
    headline = min(range(len(lstm.QUANTILES)),
                   key=lambda i: abs(lstm.QUANTILES[i] - evaluation.DEFAULT_ALPHA))
    benchmark[device_name] = {
        'wall_seconds': elapsed,
        'best_epoch': best_epoch,
        'sec_per_epoch': elapsed / epochs_run,
        'peak_rss_gb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9,
        'pred_mean_q90': float(prediction[:, headline].mean()),
        'pred_max_q90': float(prediction[:, headline].max()),
        # Head komposit tidak menjamin monotonisitas. Diukur, bukan diasumsikan.
        'crossing_rate': float(evaluation.crossing_rate(prediction, lstm.QUANTILES)),
    }
    print(device_name, benchmark[device_name])

pd.DataFrame(benchmark).T


## Anggaran pencarian

`N` datang dari angka benchmark, bukan dari tebakan. Kalau rumusnya
jatuh di bawah 6, `candidate_budget` melempar ValueError — itu sinyal
untuk memperkecil ruang search, bukan menaikkan plafon 8 jam.

In [ ]:
DEVICE = min(benchmark, key=lambda name: benchmark[name]['sec_per_epoch'])
measured = benchmark[DEVICE]

# N **dipatok** 30, setara XGBoost (spec §2.2, penyetaraan anggaran
# 2026-08-24). candidate_budget() tetap dijalankan, tetapi sebagai
# *pengukuran ongkos*, bukan sebagai penentu N: plafon 8 jam sudah
# ditinggalkan secara sadar, dan wall clock sebenarnya dicatat di
# docs/hasil-modeling-lstm.md sebagai ongkos terukur.
try:
    implied = lstm.candidate_budget(
        sec_per_epoch=measured['sec_per_epoch'],
        best_epoch=measured['best_epoch'],
    )
except ValueError as failure:
    implied = f'<{lstm.MIN_CANDIDATES} ({failure})'

N_CANDIDATES = lstm.N_CANDIDATES
print('device terpilih  :', DEVICE)
print('sec_per_epoch    :', round(measured['sec_per_epoch'], 1))
print('best_epoch       :', measured['best_epoch'])
print('N menurut plafon :', implied, '(informatif saja)')
print('N yang dipakai   :', N_CANDIDATES, '- dipatok, setara XGBoost')
print('ruang pencarian  :', np.prod([len(v) for v in lstm.SEARCH_SPACE.values()]),
      'titik')


## Pencarian hyperparameter

Fold 3 dan 5 saja, seed 42, kriteria pinball@0.9 gabungan berbobot
jumlah baris. Checkpoint di-flush tiap kandidat selesai.

In [ ]:
# Ke-30 kandidat dijalankan ulang di bawah head multi-kuantil, pada ruang
# 144 yang sudah dipulihkan (num_layers dan hidden_size kembali). Kalau
# lstm_search_results.csv dari run kuantil-tunggal masih ada, sel ini berhenti
# di menit pertama — guard checkpoint menolak berkas tanpa kolom
# `headline_quantile`. Hapus berkasnya lalu jalankan ulang.
candidates = lstm.sample_search_space(N_CANDIDATES, seed=42)
search_results = lstm.run_search(
    df, candidates,
    checkpoint_path=lstm.SEARCH_FILE,
    device_name=DEVICE,
)
# `pinball` di sini adalah K1. Kolom *_headline dibaca di tau=0,9.
search_results.sort_values('pinball').head(10)


## Walk-forward final

Pemenang dijalankan ulang di kelima fold, lalu difit final.

In [ ]:
best = lstm.select_best(search_results, candidates)
lstm.save_best_params(best)
print(best)

make = lstm.bind_panel(df, device_name=DEVICE)
fit_predict = make(best, quantiles=lstm.QUANTILES)
results = walk_forward.run_walk_forward(
    df, fit_predict, model_name='lstm', quantiles=lstm.QUANTILES)
results.to_csv(lstm.RESULTS_FILE, index=False)
print('best_epoch per fold:', fit_predict.best_epochs)
print(f"K1: {walk_forward.pooled_k1(results, 'lstm'):.4f}")


## Pengulangan tiga seed pada pemenang

LSTM satu-satunya model di perbandingan ini yang inisialisasi bobotnya acak,
jadi selisih K1 kecil antara ia dan model pohon selama ini tidak dapat
dipisahkan dari derau seed. Konfigurasi pemenang — dan hanya itu — diulang
pada seed 42/43/44 di fold pencarian yang sama (3 dan 5).

Baris seed 42 harus sama persis dengan baris pemenang di
`lstm_search_results.csv`. Kalau tidak, yang terukur bukan varians seed
melainkan nondeterminisme yang belum tertangkap — dan itu temuan tersendiri,
bukan pembulatan yang boleh dilewatkan.


In [ ]:
repeats = lstm.run_seed_repeats(
    df, best, seeds=lstm.SEED_REPEATS, folds=lstm.SEARCH_FOLDS,
    device_name=DEVICE, output_path=lstm.SEED_REPEATS_FILE)

spread = lstm.seed_spread(repeats)
print(f"K1 antar seed: min {spread['min']:.4f}  mean {spread['mean']:.4f}  "
      f"max {spread['max']:.4f}  rentang {spread['range']:.4f}")

# Konsistensi terhadap pencarian: baris seed 42 vs baris pemenang.
winner = search_results.loc[search_results['pinball'].idxmin()]
delta = float(repeats.loc[repeats['seed'] == 42, 'pinball'].iloc[0]) - float(winner['pinball'])
print(f"selisih seed-42 terhadap baris pemenang di search: {delta:+.6f} "
      f"({'konsisten' if abs(delta) < 1e-6 else 'PERIKSA - ada nondeterminisme'})")

repeats


In [ ]:
bundle = lstm.fit_final(df, best, device_name=DEVICE)
lstm.save_bundle(bundle)
print(f"best_epoch {bundle['best_epoch']}, n_train {bundle['n_train']:,}, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")


## Hasil

In [ ]:
results = pd.read_csv(lstm.RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak dirata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan
# yang berbeda. Jadi ketiganya dibaca di tau headline, eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

print("\n=== K2: coverage per titik kuantil (lstm) ===")
print(walk_forward.coverage_by_quantile(results, "lstm").round(4)
      .to_string(index=False))


## Head-to-head tiga arah

Sah karena ketiganya dinilai di baris identik — dijamin
`walk_forward.eligible_rows()`. Potongan kedua (fold 1, 2, 4) adalah
angka bersihnya: tidak ada model yang memakai fold itu untuk seleksi.

In [ ]:
from utils.modelling import model_random_forest as rf
from utils.modelling import model_xgboost as xgb

HEADLINE = evaluation.DEFAULT_ALPHA
tables = {
    'lstm': pd.read_csv(lstm.RESULTS_FILE),
    'xgboost': pd.read_csv(xgb.RESULTS_FILE),
    'random_forest': pd.read_csv(rf.RESULTS_FILE),
}
rows = []
for name, table in tables.items():
    rows.append({
        'model': name,
        'k1_5_folds': walk_forward.pooled_k1(table, name),
        'k1_folds_124': walk_forward.pooled_k1(table, name, folds=(1, 2, 4)),
        'mae_q90': walk_forward.pooled_metric(
            table, name, metric='mae', quantile=HEADLINE),
        'coverage_q90': walk_forward.pooled_metric(
            table, name, metric='coverage', quantile=HEADLINE),
        'crossing_rate': walk_forward.pooled_metric(
            table, name, metric='crossing_rate'),
    })
table = pd.DataFrame(rows).sort_values('k1_folds_124')

# Jarak K1 antar model dibaca bersama rentang antar seed di atas: kalau
# rentang seed LSTM melebihi jarak ini, jarak itu bukan perbedaan antar model.
print(f"rentang K1 antar seed (LSTM): {spread['range']:.4f}")
print(f"jarak K1 terkecil antar model: "
      f"{table['k1_folds_124'].diff().abs().min():.4f}")
table
